In [ ]:
from pathlib import Path
import geopandas as gpd
import matplotlib.pyplot as plt
import folium
try:
    display
except NameError:
    def display(value):
        print(value)

PROJECT_DIR = Path.cwd()
if PROJECT_DIR.name == "notebooks":
    PROJECT_DIR = PROJECT_DIR.parent
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
RAW_DIR = PROJECT_DIR / "data" / "raw"
OUTPUT_DIR = PROJECT_DIR / "outputs" / "maps"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

points_file = PROCESSED_DIR / "data_points.geojson"
if not points_file.exists():
    raise FileNotFoundError("Executez d'abord 02_nettoyage.ipynb pour créer data_points.geojson")

data_points = gpd.read_file(points_file)
communes = gpd.read_file(RAW_DIR / "Limites administratives - Communes.json")
if data_points.crs != communes.crs:
    communes = communes.to_crs(data_points.crs)

points_hors_limites = gpd.sjoin(data_points, communes[["geometry"]], how="left", predicate="within")
data_points["dans_commune"] = points_hors_limites["index_right"].notna().to_numpy()
print(f"Points dans une commune : {data_points['dans_commune'].mean() * 100:.1f}%")

region_summary = data_points.groupby(["region_nom_bdd", "service"], dropna=False).size().reset_index(name="points")
display(region_summary.sort_values("points", ascending=False).head(20))

fig, ax = plt.subplots(figsize=(10, 8))
communes.boundary.plot(ax=ax, linewidth=0.35, color="#71817c")
data_points.plot(ax=ax, column="service", categorical=True, legend=True, markersize=8, alpha=0.75)
ax.set_title("Points de services numériques et limites communales")
ax.set_axis_off()
plt.tight_layout()
plt.show()

map_togo = folium.Map(location=[8.7, 0.9], zoom_start=7, tiles="OpenStreetMap", control_scale=True)
for service, group in data_points.groupby("service"):
    layer = folium.FeatureGroup(name=service, show=True)
    for _, point in group.iterrows():
        folium.CircleMarker(location=[point.geometry.y, point.geometry.x], radius=4, fill=True, fill_opacity=0.8, tooltip=f"{service} - {point.get('nom', service)}", popup=f"{point.get('region_nom_bdd', '')}<br>{point.get('commune_nom_bdd', '')}").add_to(layer)
    layer.add_to(map_togo)
folium.LayerControl(collapsed=False).add_to(map_togo)
display(map_togo)
map_togo.save(OUTPUT_DIR / "services_numeriques.html")